# 🏷️ value_factor — ranking stocks by cheapness and quality

A **daily, cross-sectional** strategy built on **company fundamentals** (SEC EDGAR
XBRL `companyfacts`, point-in-time by filing date) rather than price alone. Each
day it scores every liquid name on three well-replicated factors and holds the
top-ranked slice:

- **Book-to-price** (equity ÷ market cap) — Fama-French value.
- **Earnings yield** (TTM net income ÷ market cap) — value.
- **Gross profitability** (TTM gross profit ÷ assets) — Novy-Marx quality.

Factors are combined as an **equal-weighted percentile-rank composite** and the
top `max_positions` names are held, re-selected on a monthly cadence (value and
quality are slow). One shared $10k book (portfolio mode); every entry executes at
the next bar's open and every feature is point-in-time lagged — no look-ahead.

Flow: **load → overview → backtest → signal quality → sweep → optimize → pick →
boundary → validate (baseline, cost stress, sub-period, holdout).**

> Fundamentals must be loaded first: run `notebooks/03_altdata_update.ipynb` (or
> `tsim altdata --dataset fundamentals --universe LiquidLeaders --since 2020-01-01`).

## 1 · Load the panel and attach fundamentals features

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dataclasses import replace

import numpy as np
import pandas as pd

from trading_simple.config import load_settings
from trading_simple.db import make_engine
from trading_simple import backtest as bt, features, jobs, optimize as opt, plots, strategies
from trading_simple.data import load_panel, symbols_with_bars

settings = load_settings()
engine = make_engine(settings)
strat = strategies.load("value_factor")
d = strat.data

syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]), d["bar"],
                         d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(panel, engine, d["features"], settings=settings)

print(f"{d['universe']}: {len(syms)} symbols, {panel.close.shape[0]} daily bars "
      f"{panel.close.index[0].date()} → {panel.close.index[-1].date()}")
for name, frame in panel.features.items():
    covered = int(frame.notna().any().sum())
    print(f"  feature {name:12s}: {covered}/{len(syms)} symbols with data")

## 2 · Strategy overview

The parameter table below is rendered straight from `config.yaml`, so it never
drifts from what actually runs.

In [ ]:
print("defaults (params):")
for k, v in strat.params.items():
    print(f"  {k:16s} = {v}")
print("\nsweep grid:")
for k, v in strat.sweep.items():
    print(f"  {k:16s} : {v}")
print("\noptuna search space:")
for k, v in strat.optuna["space"].items():
    print(f"  {k:16s} : {v}")

## 3 · One backtest at the defaults

Costs are always on (commission + half-spread + slippage per leg). Read
`pct_profitable` (share of names that made money) as the breadth check — a
cross-sectional strategy should win across many names, not one lucky ticker.

In [ ]:
res = bt.run_strategy(strat, panel, settings=settings)
stats0 = bt.portfolio_stats(res)
for k, v in stats0.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res)).show()

## 4 · Does the ranking predict anything at all?

**Run this before trusting any backtest number.** It asks the only question that
matters for a cross-sectional strategy — *does a higher rank mean a higher future
return?* — without the trading engine, so the answer cannot be contaminated by
exposure, costs, or exit rules.

The trick is to **subtract each day's cross-sectional mean from the forward
returns**. What is left is pure stock selection: if the whole market rises, that
rise is removed. This matters because a long-only strategy in a rising market
shows a healthy Sharpe whether or not its stock-picking works — a backtest cannot
separate the two; this can.

- **Rank IC** — correlation between today's rank and the future *relative* return.
  +0.03 upward is respectable for a slow factor; zero means noise.
- **Decile spread** — top 10% minus bottom 10%, relative to the market. Should be
  positive and grow with the horizon if the signal is real.

Each factor is scored separately, so you can see which carry the composite and
which are dead weight. A **negative** spread is a factor pulling the wrong way.

In [ ]:
HORIZONS = (21, 63, 126)          # ~1, 3 and 6 months ahead
scores = strat.signals.__globals__["scores"]   # public helper on the strategy module

def signal_quality(score: pd.DataFrame, close: pd.DataFrame) -> pd.DataFrame:
    """Rank IC and decile spread of a score against DEMEANED forward returns.

    Demeaning each day's forward returns cross-sectionally removes the market, so what
    remains measures stock selection alone. The t-stat is divided by sqrt(horizon)
    because overlapping forward windows make consecutive daily ICs highly dependent —
    the naive t-stat is inflated by ~sqrt(h).
    """
    rows = []
    for h in HORIZONS:
        fwd = close.shift(-h) / close - 1.0
        rel = fwd.sub(fwd.mean(axis=1), axis=0)          # <- the market, removed
        ic = score.corrwith(rel, axis=1, method="spearman").dropna()
        naive_t = ic.mean() / (ic.std() / np.sqrt(len(ic))) if len(ic) and ic.std() else 0.0
        r = score.rank(axis=1, pct=True)
        rows.append({
            "horizon": f"{h}d",
            "rank_IC": round(float(ic.mean()), 4),
            "t_stat (overlap-adj)": round(float(naive_t / np.sqrt(h)), 2),
            "top_decile": float(rel.where(r >= 0.9).mean(axis=1).mean()),
            "bottom_decile": float(rel.where(r <= 0.1).mean(axis=1).mean()),
        })
    out = pd.DataFrame(rows).set_index("horizon")
    out["spread"] = out["top_decile"] - out["bottom_decile"]
    return out

FMT = {"top_decile": "{:+.2%}", "bottom_decile": "{:+.2%}", "spread": "{:+.2%}"}
parts_q = scores(panel, **strat.params)
print("COMPOSITE — the score the strategy actually trades")
display(signal_quality(parts_q["score_pct"], panel.close).style.format(FMT))

print("\nPER FACTOR — who is carrying the composite, and who is dead weight?")
summary = {}
for k in parts_q["present"]:
    q = signal_quality(parts_q["ranks"][k], panel.close)
    summary[k] = {"63d IC": q.loc["63d", "rank_IC"], "63d spread": q.loc["63d", "spread"],
                  "126d IC": q.loc["126d", "rank_IC"], "126d spread": q.loc["126d", "spread"]}
display(pd.DataFrame(summary).T.style.format(
    {"63d spread": "{:+.2%}", "126d spread": "{:+.2%}",
     "63d IC": "{:+.4f}", "126d IC": "{:+.4f}"}))
print("A NEGATIVE spread is a factor pulling the composite the wrong way. And a positive")
print("spread here does NOT guarantee the backtest can harvest it — holding the top 15")
print("names, or holding them after their rank decays, dilutes the edge (see §9-10).")

## 5 · Coarse sweep

A small grid to see the shape of the objective before the optimizer runs. The
`w_*: [0.0, 1.0]` columns ablate each factor, so its marginal value is visible
side by side.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
sweep = bt.sweep(strat, panel, grid, settings=settings)
display(sweep.sort_values("excess_cagr", ascending=False).head(10).round(3))
plots.plot_sweep(sweep).show()

## 6 · Optimize (Optuna, resumable)

Maximizes `info_ratio` **and** `excess_cagr` together (a Pareto front), both
measured against equal-weight buy-and-hold — so a trial scores above zero only by
beating the market. The study is journal-backed, so this cell resumes if re-run.

In [ ]:
N_OPTIMIZE_TRIALS = 400
jobs.run_optimize("value_factor", n_trials=N_OPTIMIZE_TRIALS)

## 7 · Read the study — admissibility before picking

Few-trade parameter sets can post a flattering number by luck, so we filter the
Pareto front to trials with at least `MIN_TRADES` before choosing. Among the
admissible front we take the objective-weighted best, picking on the study's own
benchmark-relative objectives (never absolute Sharpe, which would just select the
most market-exposed combo).

In [ ]:
MIN_TRADES = 40
study = opt.load_study(strat)
front = opt.pareto_frame(study)
admissible = front[front["n_trades"] >= MIN_TRADES].copy()
print(f"front: {len(front)} trials, admissible (n_trades ≥ {MIN_TRADES}): {len(admissible)}")

_INTS = ("rebalance_days", "max_positions")
def coerce(params):
    """Cast params back to real types (front rows store everything as object/float)."""
    return {k: (int(round(v)) if k in _INTS else float(v)) for k, v in params.items()}

X, Y = opt.OBJECTIVES                       # (info_ratio, excess_cagr)
pool = admissible if len(admissible) else front
# ONE picking rule for the whole repo (opt.PICK_WEIGHTS, balanced), and it collapses tied
# outcomes first: `n_tied` > 1 says the pick was arbitrary among that many equal trials.
best_row = opt.pick_from_front(pool)
print(f"{int(best_row['front_distinct'])} distinct outcomes on the admissible front"
      + (f" — picked one of {int(best_row['n_tied'])} TIED trials, arbitrarily"
         if best_row["n_tied"] > 1 else ""))
picked = coerce({k: best_row[k] for k in strat.optuna["space"]})

beats = (front[X] > 0) & (front[Y] > 0)
print(f"trials on the front that BEAT buy-and-hold: {int(beats.sum())} of {len(front)}")
print("picked params:", picked)
print("picked train metrics:", {m: round(float(best_row[m]), 4)
      for m in (X, Y, "sharpe", "cagr", "pct_profitable", "n_trades",
                "trades_per_year", "years")})
plots.plot_pareto(front, opt.trials_frame(study), picked=int(best_row["trial"])).show()

## 8 · Boundary check

If a picked value sits on a search-space edge, the real optimum may lie outside
the box — widen that bound and re-run. (Edge flags driven only by the thin
few-trade tail are false positives; judge against the admissible front.)

In [ ]:
display(opt.boundary_report(picked, strat.optuna["space"], front=front))

## 9 · Backtest the picked parameters

In [ ]:
res_best = bt.run_strategy(strat, panel, params=picked, settings=settings)
stats_best = bt.portfolio_stats(res_best)
for k, v in stats_best.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(res_best)).show()
display(bt.symbol_stats(res_best).sort_values("total_return", ascending=False).head(10).round(3))

## 10 · Validate — is the edge real?

Everything above is in-sample. **§10a is the one that can kill the strategy
outright** — a long-only ranker earns a positive absolute Sharpe merely by being
invested, so the only question that counts is whether it beats simply owning the
universe.

In [ ]:
def slice_panel(p, start, end):
    """A copy of the panel restricted to [start, end], features sliced too."""
    m = ((p.close.index >= pd.Timestamp(start, tz="UTC"))
         & (p.close.index <= pd.Timestamp(end, tz="UTC")))
    idx = p.close.index[m]
    feats = {k: v.loc[idx] for k, v in (p.features or {}).items()}
    return replace(p, close=p.close.loc[idx], open_=p.open_.loc[idx], high=p.high.loc[idx],
                   low=p.low.loc[idx], volume=p.volume.loc[idx], features=feats)

### 10a · The hurdle: does it beat simply owning the universe?

The reference is an **equal-weight buy-and-hold** of the same names over the same
period, paying the same costs. If the strategy does not beat it, the stock
selection is adding nothing — regardless of how healthy the headline looks. This
is also what the optimizer maximized (`info_ratio` and `excess_cagr` against this
benchmark).

In [ ]:
def hurdle(p, params, label):
    """Strategy vs equal-weight buy-and-hold of the same panel. The pass/fail gate."""
    st = bt.evaluate(strat, p, params, settings=settings)
    fees, slip = bt.panel_costs(settings.costs, p)
    bench_ret, _ = bt.benchmark(p, fees=fees, slippage=slip)
    bench_cagr = bt._cagr(bench_ret, st["years"])
    ok = st["excess_cagr"] > 0 and st["info_ratio"] > 0
    verdict = ("PASS — it beats owning the universe" if ok
               else "FAIL — owning the universe was better")
    print(f"\n{label}  ({p.close.index[0].date()} -> {p.close.index[-1].date()}, "
          f"{st['years']:.2f} years, {st['trades_per_year']:.0f} trades/yr)")
    print(f"  strategy     {st['cagr']:+7.2%}/yr   sharpe {st['sharpe']:5.2f}   "
          f"n_trades {int(st['n_trades'])}")
    print(f"  buy & hold   {bench_cagr:+7.2%}/yr")
    print(f"  EXCESS       {st['excess_cagr']:+7.2%}/yr   "
          f"information ratio {st['info_ratio']:+5.2f}")
    print(f"  --> {verdict}")
    return st

hurdle(panel, picked, "TRAIN")
full_h = load_panel(engine, syms, d["bar"], d["start"], None,
                    session=d.get("session", "regular"), exchanges=settings.exchanges)
features.attach_features(full_h, engine, d["features"], settings=settings)
holdout = slice_panel(full_h, "2025-01-01", str(full_h.close.index[-1].date()))
hurdle(holdout, picked, "HOLDOUT (2025+)")

### 10b · Costs on, then doubled — and the breakeven

Never trust a zero-cost result. Re-run the pick at 1×/2×/3× the configured costs,
then find the round-trip cost (in bps) at which the Sharpe hits zero. A monthly
rebalance is cheap on costs, but confirm the edge is not a cost-hurdle mirage.

In [ ]:
def with_costs(**overrides):
    """A copy of settings with its (pydantic) cost block overridden."""
    return settings.model_copy(update={"costs": settings.costs.model_copy(update=overrides)})

c = settings.costs
for mult in (1.0, 2.0, 3.0):
    s = with_costs(spread_bps=c.spread_bps * mult, slippage_bps=c.slippage_bps * mult,
                   commission_per_share=c.commission_per_share * mult)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    print(f"  {mult:.0f}x costs: sharpe={st['sharpe']:.3f}  return={st['total_return']:.2%}")

base_bps = c.spread_bps + c.slippage_bps
breakeven = None
for extra in np.arange(0, 60, 2.0):
    s = with_costs(spread_bps=base_bps + extra, slippage_bps=0.0)
    st = bt.portfolio_stats(bt.run_strategy(strat, panel, params=picked, settings=s))
    if st["sharpe"] <= 0:
        breakeven = base_bps + extra
        break
be = ">60" if breakeven is None else round(breakeven, 1)
print(f"\napprox breakeven half-spread: {be} bps per leg")

### 10c · Sub-period consistency

One number over four years can hide a strategy that only worked in one regime.
Score the pick on each calendar year and read the table, not a pass/fail.

In [ ]:
years = []
metrics = ("total_return", "excess_cagr", "sharpe", "pct_profitable", "n_trades")
for y in range(2021, 2025):
    sub = slice_panel(panel, f"{y}-01-01", f"{y}-12-31")
    st = bt.evaluate(strat, sub, picked, settings=settings)
    years.append({"year": y, **{m: round(float(st[m]), 3) for m in metrics}})
display(pd.DataFrame(years).set_index("year"))

### 10d · Untouched holdout (open once)

2025+ was never seen by the config, the sweep, or the optimizer. Score the pick on
it exactly once. If it disappoints, that is information — do not tune against it.

In [ ]:
st = bt.portfolio_stats(bt.run_strategy(strat, holdout, params=picked, settings=settings))
print(f"HOLDOUT {holdout.close.index[0].date()} → {holdout.close.index[-1].date()} "
      f"({holdout.close.shape[0]} bars):")
for k, v in st.items():
    print(f"  {k:16s} = {v:.4f}")
plots.plot_equity(bt.equity_curve(bt.run_strategy(strat, holdout, params=picked,
                                                  settings=settings))).show()

## Read the result honestly

- The **holdout** and **baseline** numbers are the real ones; the in-sample and
  picked metrics are not.
- Judge breadth on **`pct_profitable`**, not one big winner, and always read
  **`n_trades`** next to any Sharpe.
- **§4 rank-IC** is the cleanest read on whether the *signal* works; **§10a** is
  whether the *strategy* beats owning the universe. Both must hold.
- ~4 years is a short sample, and value/quality factors go through long cold
  spells — prefer consistency across §10c to a high headline.
- Foreign 20-F names carry only annual fundamentals (TTM degrades to last fiscal
  year) and warm up ~a year after `data.start`; keep claims to this liquid universe.